# Serpy — Advanced Problems with Complete Solutions

This notebook is a **problem-driven deep dive** into `serpy`, a small Python library for **serialization**: turning Python objects (or dictionaries) into native Python data such as `dict`, `list`, `str`, `int`, `float`, and `bool`.

> **Important correction:** Serpy is **serialization-only**. It does **not** deserialize input into objects and does **not** provide input validation via `Serializer(data=...)`. Use another library when you need parsing/deserialization/validation.

## What you will practice

- Basic and bulk serialization with `many=True`
- Nested serializers
- Attribute remapping with `attr=`
- Output renaming with `label=`
- Calling object methods with `call=True`
- Computed fields with `MethodField`
- Optional and missing fields with `required=False`
- Custom `Field` classes
- `DictSerializer`
- Serializer inheritance and API versioning
- Deeply nested object graphs
- Avoiding circular references
- Stable JSON/YAML output
- Security-oriented allowlists and redaction
- Understanding `.data` caching
- Micro-benchmarking
- A larger capstone exercise

Every problem is followed by a complete solution and executable assertions.


## Setup

The original `serpy` package's PyPI release is `0.3.1`. It is intentionally minimal and old, so pinning the version makes examples reproducible.

If your environment already has the package, you may skip the installation cell.


In [ ]:
%pip install -q serpy==0.3.1 PyYAML


In [ ]:
from __future__ import annotations

import json
import timeit
from dataclasses import dataclass, field
from datetime import date, datetime, timezone
from decimal import Decimal
from enum import Enum
from typing import Any

import serpy
import yaml

print("serpy imported successfully")


## Quick reference

A serializer is a class whose fields describe which values are emitted:

```python
class ExampleSerializer(serpy.Serializer):
    name = serpy.StrField()
    age = serpy.IntField()
```

Core field options:

- `attr="source.path"` — read from a differently named attribute; dotted paths work through `operator.attrgetter`.
- `label="output_name"` — rename the key in serialized output.
- `call=True` — call the fetched value as a zero-argument callable.
- `required=False` — tolerate a missing attribute/key and omit it; if the value is `None`, skip conversion and preserve `None`.
- `many=True` — serialize a collection.
- `MethodField()` — compute a field using a serializer method named `get_<field>()`.


# Part 1 — Core serialization


## Problem 1 — Repair and extend the original `Person` example

Create a `Person` model and a serializer that:

1. Emits `name` as a string.
2. Emits `age` as an integer.
3. Serializes one object.
4. Serializes a list with `many=True`.
5. Demonstrates that Serpy fields can coerce compatible values, such as `"42"` to `42`.


In [ ]:
@dataclass
class Person:
    name: str
    age: Any


class PersonSerializer(serpy.Serializer):
    name = serpy.StrField()
    age = serpy.IntField()


p1 = Person("Michael Palin", 75)
p2 = Person("John Cleese", "79")

one_person = PersonSerializer(p1).data
many_people = PersonSerializer([p1, p2], many=True).data

print(one_person)
print(many_people)

assert one_person == {"name": "Michael Palin", "age": 75}
assert many_people[1]["age"] == 79
assert isinstance(many_people[1]["age"], int)


### Solution notes

`IntField` calls `int(...)` during serialization. This is **output conversion**, not input validation. If conversion fails, the exception propagates.


## Problem 2 — Nested serializers and `many=True`

Model a movie with a title, year, and a list of actors. Serialize the actors as nested objects.


In [ ]:
@dataclass
class Movie:
    title: str
    year: int
    actors: list[Person]


class MovieSerializer(serpy.Serializer):
    title = serpy.StrField()
    year = serpy.IntField()
    actors = PersonSerializer(many=True)


movie = Movie(
    title="Parrot Sketch",
    year=1989,
    actors=[p1, p2],
)

movie_data = MovieSerializer(movie).data
print(json.dumps(movie_data, indent=2))

assert movie_data["title"] == "Parrot Sketch"
assert movie_data["actors"][0]["name"] == "Michael Palin"
assert movie_data["actors"][1]["age"] == 79


## Problem 3 — Rename both the source attribute and the output key

Suppose an internal model uses awkward names:

- `_display_name` internally, but the API must emit `name`
- `birth_year` internally, but the API must emit `year_of_birth`

Use `attr=` and `label=` without changing the model.


In [ ]:
@dataclass
class LegacyUser:
    _display_name: str
    birth_year: int


class LegacyUserSerializer(serpy.Serializer):
    display_name = serpy.StrField(
        attr="_display_name",
        label="name",
    )
    birth = serpy.IntField(
        attr="birth_year",
        label="year_of_birth",
    )


legacy = LegacyUser("Ada", 1815)
legacy_data = LegacyUserSerializer(legacy).data

print(legacy_data)

assert legacy_data == {
    "name": "Ada",
    "year_of_birth": 1815,
}


### Best-practice pattern

Use `attr=` to decouple your external API contract from internal model names. Use `label=` when the output key should differ from the serializer field name.


## Problem 4 — Traverse dotted attributes

Serialize `account.profile.handle` without adding proxy properties to the model.


In [ ]:
@dataclass
class Profile:
    handle: str


@dataclass
class Account:
    profile: Profile
    active: bool


class AccountSerializer(serpy.Serializer):
    username = serpy.StrField(
        attr="profile.handle",
        label="username",
    )
    active = serpy.BoolField()


account = Account(Profile("deep_python"), True)
account_data = AccountSerializer(account).data

print(account_data)

assert account_data == {
    "username": "deep_python",
    "active": True,
}


# Part 2 — Methods, computed fields, and optional values


## Problem 5 — Serialize a zero-argument object method with `call=True`

Create a product whose model has a `display_price()` method. Emit the method's return value as `display_price`.


In [ ]:
@dataclass
class Product:
    sku: str
    unit_price: Decimal

    def display_price(self) -> str:
        return f"${self.unit_price:.2f}"


class ProductSerializer(serpy.Serializer):
    sku = serpy.StrField()
    display_price = serpy.StrField(call=True)


product = Product("SKU-100", Decimal("19.995"))
product_data = ProductSerializer(product).data

print(product_data)

assert product_data == {
    "sku": "SKU-100",
    "display_price": "$20.00",
}


### When to use `call=True`

Use it when the value already exists as a **zero-argument method on the object**.

If a result depends on several attributes or belongs to presentation logic, `MethodField` is usually clearer.


## Problem 6 — Compute fields with `MethodField`

Serialize a cart line with:

- `quantity`
- `unit_price`
- computed `subtotal`

Keep monetary values JSON-safe by returning strings rather than binary floating-point values.


In [ ]:
@dataclass
class CartLine:
    sku: str
    quantity: int
    unit_price: Decimal


class CartLineSerializer(serpy.Serializer):
    sku = serpy.StrField()
    quantity = serpy.IntField()
    unit_price = serpy.MethodField()
    subtotal = serpy.MethodField()

    def get_unit_price(self, obj: CartLine) -> str:
        return format(obj.unit_price, ".2f")

    def get_subtotal(self, obj: CartLine) -> str:
        return format(obj.unit_price * obj.quantity, ".2f")


line = CartLine("ABC-9", 3, Decimal("12.50"))
line_data = CartLineSerializer(line).data

print(line_data)

assert line_data["unit_price"] == "12.50"
assert line_data["subtotal"] == "37.50"


## Problem 7 — Use a custom `MethodField` method name

Create `total` using a method named `calculate_total_for_output` instead of the default `get_total`.


In [ ]:
class ExplicitMethodSerializer(serpy.Serializer):
    total = serpy.MethodField("calculate_total_for_output")

    def calculate_total_for_output(self, obj: CartLine) -> str:
        return format(obj.quantity * obj.unit_price, ".2f")


explicit_method_data = ExplicitMethodSerializer(line).data
print(explicit_method_data)

assert explicit_method_data == {"total": "37.50"}


## Problem 8 — Optional fields and `required=False`

Create a serializer where `nickname` is optional.

Test all three cases:

1. Attribute exists with a string.
2. Attribute exists with `None`.
3. Attribute does not exist at all.

Observe the subtle difference between `None` and a missing attribute.


In [ ]:
class FlexibleUser:
    def __init__(self, name: str, nickname_marker: Any = ...):
        self.name = name
        if nickname_marker is not ...:
            self.nickname = nickname_marker


class FlexibleUserSerializer(serpy.Serializer):
    name = serpy.StrField()
    nickname = serpy.StrField(required=False)


with_nickname = FlexibleUser("Alice", "Al")
with_none = FlexibleUser("Bob", None)
without_attribute = FlexibleUser("Carol")

optional_results = FlexibleUserSerializer(
    [with_nickname, with_none, without_attribute],
    many=True,
).data

print(optional_results)

assert optional_results[0] == {"name": "Alice", "nickname": "Al"}
assert optional_results[1] == {"name": "Bob", "nickname": None}
assert optional_results[2] == {"name": "Carol"}


### Key detail

With `required=False`:

- a **missing attribute/key** is omitted;
- an attribute/key whose value is **`None`** is retained as `None`;
- field conversion is skipped for that `None`.

That behavior is useful, but it is not a replacement for a validation layer.


# Part 3 — Custom fields


## Problem 9 — Create an ISO date field

Create a reusable field that serializes `date` and `datetime` values using `.isoformat()` and rejects unsupported values.


In [ ]:
class ISODateField(serpy.Field):
    def to_value(self, value: Any) -> str:
        if isinstance(value, (date, datetime)):
            return value.isoformat()
        raise TypeError(
            f"ISODateField expected date/datetime, got {type(value).__name__}"
        )


@dataclass
class Release:
    name: str
    released_on: date


class ReleaseSerializer(serpy.Serializer):
    name = serpy.StrField()
    released_on = ISODateField()


release = Release("v2", date(2026, 8, 7))
release_data = ReleaseSerializer(release).data

print(release_data)

assert release_data == {
    "name": "v2",
    "released_on": "2026-08-07",
}


## Problem 10 — Serialize `Decimal` safely

Create a `DecimalStringField` that emits an exact decimal string.

Why string output? JSON numbers do not preserve Python `Decimal` semantics by themselves, and converting money to `float` can introduce binary floating-point artifacts.


In [ ]:
class DecimalStringField(serpy.Field):
    def to_value(self, value: Any) -> str:
        if not isinstance(value, Decimal):
            value = Decimal(str(value))
        return format(value, "f")


@dataclass
class Quote:
    symbol: str
    price: Decimal


class QuoteSerializer(serpy.Serializer):
    symbol = serpy.StrField()
    price = DecimalStringField()


quote = Quote("XYZ", Decimal("12345.6700"))
quote_data = QuoteSerializer(quote).data

print(quote_data)

assert quote_data == {
    "symbol": "XYZ",
    "price": "12345.6700",
}


## Problem 11 — Serialize `Enum` values with a reusable field


In [ ]:
class EnumValueField(serpy.Field):
    def to_value(self, value: Enum) -> Any:
        if not isinstance(value, Enum):
            raise TypeError("EnumValueField expected an Enum instance")
        return value.value


class OrderStatus(Enum):
    PENDING = "pending"
    PAID = "paid"
    CANCELLED = "cancelled"


@dataclass
class SmallOrder:
    order_id: int
    status: OrderStatus


class SmallOrderSerializer(serpy.Serializer):
    order_id = serpy.IntField()
    status = EnumValueField()


small_order = SmallOrder(501, OrderStatus.PAID)
small_order_data = SmallOrderSerializer(small_order).data

print(small_order_data)

assert small_order_data == {
    "order_id": 501,
    "status": "paid",
}


## Problem 12 — Advanced custom getter

A custom field can override `as_getter()` when transformation alone is not enough.

Build a field that serializes the **length** of an attribute without adding a property to every model.


In [ ]:
class LengthField(serpy.Field):
    def as_getter(self, serializer_field_name, serializer_cls):
        source_name = self.attr or serializer_field_name

        def get_length(obj):
            value = getattr(obj, source_name)
            return len(value)

        return get_length


@dataclass
class Article:
    title: str
    tags: list[str]


class ArticleSerializer(serpy.Serializer):
    title = serpy.StrField()
    tag_count = LengthField(attr="tags")


article = Article("Serialization Patterns", ["python", "api", "serpy"])
article_data = ArticleSerializer(article).data

print(article_data)

assert article_data == {
    "title": "Serialization Patterns",
    "tag_count": 3,
}


# Part 4 — Dictionaries, inheritance, and API contracts


## Problem 13 — Serialize dictionaries with `DictSerializer`

Imagine rows coming from a cache, SQL adapter, CSV-normalization step, or another system that already emits dictionaries.

Serialize a list of rows while converting numeric strings.


In [ ]:
class MetricRowSerializer(serpy.DictSerializer):
    name = serpy.StrField()
    count = serpy.IntField()
    ratio = serpy.FloatField()


rows = [
    {"name": "requests", "count": "1200", "ratio": "0.97"},
    {"name": "errors", "count": "9", "ratio": "0.0075"},
]

metric_data = MetricRowSerializer(rows, many=True).data

print(metric_data)

assert metric_data[0]["count"] == 1200
assert metric_data[0]["ratio"] == 0.97
assert isinstance(metric_data[1]["count"], int)


## Problem 14 — Optional dictionary keys

Use `required=False` to omit an absent key from a `DictSerializer`.


In [ ]:
class SparseRowSerializer(serpy.DictSerializer):
    id = serpy.IntField()
    note = serpy.StrField(required=False)


sparse_rows = [
    {"id": 1, "note": "present"},
    {"id": 2, "note": None},
    {"id": 3},
]

sparse_data = SparseRowSerializer(sparse_rows, many=True).data
print(sparse_data)

assert sparse_data == [
    {"id": 1, "note": "present"},
    {"id": 2, "note": None},
    {"id": 3},
]


## Problem 15 — Reuse fields through inheritance

Create a base serializer for shared audit fields, then build a public serializer that adds resource-specific fields.


In [ ]:
@dataclass
class Document:
    id: int
    created_at: datetime
    title: str


class AuditSerializer(serpy.Serializer):
    id = serpy.IntField()
    created_at = ISODateField()


class DocumentSerializer(AuditSerializer):
    title = serpy.StrField()


document = Document(
    id=7,
    created_at=datetime(2026, 8, 7, 12, 30, tzinfo=timezone.utc),
    title="Quarterly Report",
)

document_data = DocumentSerializer(document).data
print(document_data)

assert document_data["id"] == 7
assert document_data["title"] == "Quarterly Report"
assert document_data["created_at"].endswith("+00:00")


## Problem 16 — Version an API serializer without mutating the model

Create `CustomerV1Serializer` and `CustomerV2Serializer`.

- V1 emits `name`.
- V2 keeps the old data but additionally emits `display_name` and a computed `is_adult`.
- Do not add API-specific fields to the domain object.


In [ ]:
@dataclass
class Customer:
    id: int
    name: str
    age: int


class CustomerV1Serializer(serpy.Serializer):
    id = serpy.IntField()
    name = serpy.StrField()


class CustomerV2Serializer(CustomerV1Serializer):
    display_name = serpy.StrField(attr="name")
    is_adult = serpy.MethodField()

    def get_is_adult(self, obj: Customer) -> bool:
        return obj.age >= 18


customer = Customer(42, "Grace Hopper", 85)

v1 = CustomerV1Serializer(customer).data
v2 = CustomerV2Serializer(customer).data

print("v1:", v1)
print("v2:", v2)

assert v1 == {"id": 42, "name": "Grace Hopper"}
assert v2 == {
    "id": 42,
    "name": "Grace Hopper",
    "display_name": "Grace Hopper",
    "is_adult": True,
}


# Part 5 — Complex object graphs


## Problem 17 — Deeply nested order serialization

Build an order containing:

- a customer
- multiple order lines
- a product on every line
- quantities and line totals
- an order total

Requirements:

- no `Decimal` should leak into the final dictionary;
- all dates must be ISO formatted;
- nested collections must use nested serializers;
- money must be represented as strings.


In [ ]:
@dataclass
class CatalogProduct:
    sku: str
    name: str
    price: Decimal


@dataclass
class OrderLine:
    product: CatalogProduct
    quantity: int


@dataclass
class Order:
    id: int
    customer: Customer
    placed_at: datetime
    lines: list[OrderLine]


class CatalogProductSerializer(serpy.Serializer):
    sku = serpy.StrField()
    name = serpy.StrField()
    price = DecimalStringField()


class OrderLineSerializer(serpy.Serializer):
    product = CatalogProductSerializer()
    quantity = serpy.IntField()
    line_total = serpy.MethodField()

    def get_line_total(self, obj: OrderLine) -> str:
        return format(obj.product.price * obj.quantity, ".2f")


class OrderCustomerSerializer(serpy.Serializer):
    id = serpy.IntField()
    name = serpy.StrField()


class OrderSerializer(serpy.Serializer):
    id = serpy.IntField()
    customer = OrderCustomerSerializer()
    placed_at = ISODateField()
    lines = OrderLineSerializer(many=True)
    total = serpy.MethodField()

    def get_total(self, obj: Order) -> str:
        value = sum(
            (line.product.price * line.quantity for line in obj.lines),
            start=Decimal("0"),
        )
        return format(value, ".2f")


In [ ]:
keyboard = CatalogProduct("KB-1", "Mechanical Keyboard", Decimal("89.90"))
mouse = CatalogProduct("MS-2", "Wireless Mouse", Decimal("34.50"))

complex_order = Order(
    id=9001,
    customer=Customer(101, "Lin", 31),
    placed_at=datetime(2026, 8, 7, 16, 0, tzinfo=timezone.utc),
    lines=[
        OrderLine(keyboard, 2),
        OrderLine(mouse, 1),
    ],
)

complex_order_data = OrderSerializer(complex_order).data

print(json.dumps(complex_order_data, indent=2))

assert complex_order_data["total"] == "214.30"
assert complex_order_data["lines"][0]["line_total"] == "179.80"
assert complex_order_data["lines"][0]["product"]["price"] == "89.90"
assert json.dumps(complex_order_data)  # proves JSON encoder accepts the final structure


## Problem 18 — Avoid circular references

A common ORM-style graph is:

`Author -> books -> author -> books -> ...`

If both serializers fully nest each other, recursion never has a natural boundary.

Design a safe public representation:

- `AuthorSerializer` may include lightweight books.
- `BookSerializer` should emit `author_id` rather than nesting the full author.


In [ ]:
@dataclass
class Author:
    id: int
    name: str
    books: list["Book"] = field(default_factory=list)


@dataclass
class Book:
    id: int
    title: str
    author: Author | None = None


class BookSummarySerializer(serpy.Serializer):
    id = serpy.IntField()
    title = serpy.StrField()


class AuthorSerializer(serpy.Serializer):
    id = serpy.IntField()
    name = serpy.StrField()
    books = BookSummarySerializer(many=True)


class BookSerializer(serpy.Serializer):
    id = serpy.IntField()
    title = serpy.StrField()
    author_id = serpy.IntField(attr="author.id")


author = Author(1, "Example Author")
book_a = Book(10, "First Book", author)
book_b = Book(11, "Second Book", author)
author.books.extend([book_a, book_b])

author_payload = AuthorSerializer(author).data
book_payload = BookSerializer(book_a).data

print(author_payload)
print(book_payload)

assert author_payload["books"][0] == {"id": 10, "title": "First Book"}
assert book_payload["author_id"] == 1


### Design lesson

Serialization boundaries are part of API design. Avoid mirroring an entire in-memory object graph just because it exists. Prefer deliberate, finite representations.


## Problem 19 — Polymorphic event payloads

Suppose a stream contains heterogeneous events. A serializer cannot dynamically replace its declared schema per object, but `MethodField` can safely create a finite payload representation.

Serialize these event types:

- `user.created`
- `order.paid`


In [ ]:
@dataclass
class Event:
    event_id: str
    event_type: str
    created_at: datetime
    payload: Any


@dataclass
class UserCreatedPayload:
    user_id: int
    email: str


@dataclass
class OrderPaidPayload:
    order_id: int
    amount: Decimal


class UserCreatedPayloadSerializer(serpy.Serializer):
    user_id = serpy.IntField()
    email = serpy.StrField()


class OrderPaidPayloadSerializer(serpy.Serializer):
    order_id = serpy.IntField()
    amount = DecimalStringField()


class EventSerializer(serpy.Serializer):
    event_id = serpy.StrField()
    event_type = serpy.StrField()
    created_at = ISODateField()
    payload = serpy.MethodField()

    def get_payload(self, obj: Event) -> dict[str, Any]:
        if obj.event_type == "user.created":
            return UserCreatedPayloadSerializer(obj.payload).data
        if obj.event_type == "order.paid":
            return OrderPaidPayloadSerializer(obj.payload).data
        raise ValueError(f"Unsupported event type: {obj.event_type}")


In [ ]:
events = [
    Event(
        "evt-1",
        "user.created",
        datetime(2026, 8, 7, 10, 0, tzinfo=timezone.utc),
        UserCreatedPayload(7, "user@example.com"),
    ),
    Event(
        "evt-2",
        "order.paid",
        datetime(2026, 8, 7, 10, 1, tzinfo=timezone.utc),
        OrderPaidPayload(9001, Decimal("214.30")),
    ),
]

event_data = EventSerializer(events, many=True).data
print(json.dumps(event_data, indent=2))

assert event_data[0]["payload"]["user_id"] == 7
assert event_data[1]["payload"]["amount"] == "214.30"


# Part 6 — Output formats and API safety


## Problem 20 — Produce stable JSON and YAML

Serialize `complex_order` and then:

1. Produce pretty JSON.
2. Produce compact JSON.
3. Produce safe YAML.
4. Round-trip the JSON string back to a Python dictionary and compare it with the Serpy result.


In [ ]:
order_native = OrderSerializer(complex_order).data

pretty_json = json.dumps(
    order_native,
    indent=2,
    sort_keys=True,
)

compact_json = json.dumps(
    order_native,
    separators=(",", ":"),
    sort_keys=True,
)

safe_yaml = yaml.safe_dump(
    order_native,
    sort_keys=False,
)

round_tripped = json.loads(pretty_json)

print("PRETTY JSON")
print(pretty_json)
print("\nCOMPACT JSON")
print(compact_json)
print("\nYAML")
print(safe_yaml)

assert round_tripped == order_native
assert "\n" not in compact_json


### Best practices for format boundaries

Serpy's responsibility should usually end at **native Python data**.

Then hand that data to the boundary-specific encoder:

- `json.dumps(...)` for JSON
- `yaml.safe_dump(...)` for YAML
- a web framework's JSON response utility for HTTP responses

Keeping these layers separate makes testing easier.


## Problem 21 — Prevent accidental secret leakage

A domain object contains a password hash, reset token, and internal risk score.

Build a **public allowlist serializer** that emits only:

- `id`
- `email`
- `display_name`

Then verify forbidden fields are absent.


In [ ]:
@dataclass
class UserRecord:
    id: int
    email: str
    display_name: str
    password_hash: str
    reset_token: str
    internal_risk_score: float


class PublicUserSerializer(serpy.Serializer):
    id = serpy.IntField()
    email = serpy.StrField()
    display_name = serpy.StrField()


user_record = UserRecord(
    id=88,
    email="person@example.com",
    display_name="Person",
    password_hash="not-for-output",
    reset_token="also-secret",
    internal_risk_score=0.92,
)

public_user = PublicUserSerializer(user_record).data
print(public_user)

assert set(public_user) == {"id", "email", "display_name"}
assert "password_hash" not in public_user
assert "reset_token" not in public_user
assert "internal_risk_score" not in public_user


### Security lesson

Prefer **explicit allowlists**. A serializer that declares only public fields is safer than dumping `obj.__dict__` and trying to delete sensitive keys afterward.


## Problem 22 — Redact part of a value with a custom field

Create an email-masking field. For `alice@example.com`, emit `a***@example.com`.


In [ ]:
class MaskedEmailField(serpy.Field):
    def to_value(self, value: str) -> str:
        local, sep, domain = value.partition("@")
        if not sep:
            return "***"
        if not local:
            return f"***@{domain}"
        return f"{local[0]}***@{domain}"


class RedactedUserSerializer(serpy.Serializer):
    id = serpy.IntField()
    email = MaskedEmailField()


redacted = RedactedUserSerializer(user_record).data
print(redacted)

assert redacted == {
    "id": 88,
    "email": "p***@example.com",
}


# Part 7 — Failure modes and behavioral details


## Problem 23 — Show why Serpy is not a deserializer

Attempt to construct a serializer with `data=...` and capture the error.


In [ ]:
try:
    PersonSerializer(data={"name": "Alice", "age": 30})
except RuntimeError as exc:
    print(type(exc).__name__ + ":", exc)
    assert "do not support input validation" in str(exc)
else:
    raise AssertionError("Expected Serpy to reject data=...")


### Consequence

Use Serpy when you already have trusted/constructed Python objects and want fast output serialization.

For untrusted inbound JSON, use a validation/deserialization layer before objects reach your domain logic.


## Problem 24 — Understand `.data` caching

Serpy caches `.data` on the serializer instance.

Demonstrate that mutating the source object **after the first `.data` access** does not change the cached result. Then create a new serializer to observe the mutation.


In [ ]:
mutable_person = Person("Before", 20)
serializer_instance = PersonSerializer(mutable_person)

first = serializer_instance.data
mutable_person.name = "After"
second = serializer_instance.data
fresh = PersonSerializer(mutable_person).data

print("first :", first)
print("second:", second)
print("fresh :", fresh)

assert first is second
assert second["name"] == "Before"
assert fresh["name"] == "After"


### Best practice

Treat a serializer instance as a short-lived rendering object. Do not keep and reuse the same instance after mutating the source object.


## Problem 25 — Type conversion failure

`IntField` converts with `int(...)`. Show the failure mode when an incompatible value is encountered.


In [ ]:
bad_person = Person("Broken Age", "not-an-integer")

try:
    PersonSerializer(bad_person).data
except ValueError as exc:
    print("Expected conversion failure:", exc)
else:
    raise AssertionError("Expected ValueError")


### Interpretation

This exception can help detect unexpected domain state, but it is not a friendly user-input validation system. Validation errors should normally be produced earlier, before serialization.


# Part 8 — Performance exercises


## Problem 26 — Benchmark Serpy without making misleading claims

Benchmark two ways to serialize 10,000 simple objects:

1. `PersonSerializer(..., many=True)`
2. A hand-written list comprehension

Run several repetitions and print the best time.

**Goal:** learn how to benchmark your own workload. Do not assume one universal winner across Python versions, object shapes, interpreters, or alternative libraries.


In [ ]:
benchmark_people = [
    Person(f"Person {i}", i % 100)
    for i in range(10_000)
]


def serialize_with_serpy():
    return PersonSerializer(benchmark_people, many=True).data


def serialize_manually():
    return [
        {"name": str(person.name), "age": int(person.age)}
        for person in benchmark_people
    ]


# Warm up and verify equivalent output.
serpy_result = serialize_with_serpy()
manual_result = serialize_manually()
assert serpy_result == manual_result

serpy_times = timeit.repeat(
    serialize_with_serpy,
    number=10,
    repeat=5,
)

manual_times = timeit.repeat(
    serialize_manually,
    number=10,
    repeat=5,
)

print(f"Serpy best : {min(serpy_times):.6f}s")
print(f"Manual best: {min(manual_times):.6f}s")
print("Benchmark on your own production-shaped data before choosing.")


## Problem 27 — Benchmark JSON encoding separately

Serialization and JSON encoding are different stages. Measure them independently.


In [ ]:
native_people = PersonSerializer(benchmark_people, many=True).data


def serpy_only():
    return PersonSerializer(benchmark_people, many=True).data


def json_only():
    return json.dumps(native_people)


def serpy_plus_json():
    return json.dumps(PersonSerializer(benchmark_people, many=True).data)


print(
    "serpy_only     :",
    min(timeit.repeat(serpy_only, number=10, repeat=3)),
)
print(
    "json_only      :",
    min(timeit.repeat(json_only, number=10, repeat=3)),
)
print(
    "serpy_plus_json:",
    min(timeit.repeat(serpy_plus_json, number=10, repeat=3)),
)


### Performance lesson

Measure the pipeline you actually care about:

`domain objects -> native data -> JSON bytes/text -> network`

Optimizing only one stage may not improve end-to-end latency.


# Part 9 — Capstone


## Problem 28 — Production-style invoice payload

Design a serialization layer for an invoice system.

### Domain requirements

An invoice has:

- `invoice_id`
- `issued_at`
- customer data
- line items
- tax rate
- internal notes
- payment status

Each line item has:

- SKU
- description
- quantity
- unit price

### API requirements

Emit:

```text
invoice_id
issued_at
customer { id, name }
status
items [
    { sku, description, quantity, unit_price, subtotal }
]
subtotal
tax
total
```

Constraints:

1. Never serialize `internal_notes`.
2. All money values must be exact decimal strings with two places.
3. Dates must use ISO 8601.
4. `status` must emit the enum's value.
5. Totals must be computed from line items.
6. The final result must pass `json.dumps(...)`.


In [ ]:
class PaymentStatus(Enum):
    UNPAID = "unpaid"
    PAID = "paid"
    VOID = "void"


@dataclass
class InvoiceCustomer:
    id: int
    name: str
    billing_email: str


@dataclass
class InvoiceItem:
    sku: str
    description: str
    quantity: int
    unit_price: Decimal


@dataclass
class Invoice:
    invoice_id: str
    issued_at: datetime
    customer: InvoiceCustomer
    items: list[InvoiceItem]
    tax_rate: Decimal
    status: PaymentStatus
    internal_notes: str


### Solution


In [ ]:
class Money2Field(serpy.Field):
    def to_value(self, value: Any) -> str:
        decimal_value = value if isinstance(value, Decimal) else Decimal(str(value))
        return format(decimal_value.quantize(Decimal("0.01")), ".2f")


class InvoiceCustomerSerializer(serpy.Serializer):
    id = serpy.IntField()
    name = serpy.StrField()


class InvoiceItemSerializer(serpy.Serializer):
    sku = serpy.StrField()
    description = serpy.StrField()
    quantity = serpy.IntField()
    unit_price = Money2Field()
    subtotal = serpy.MethodField()

    def get_subtotal(self, obj: InvoiceItem) -> str:
        value = obj.unit_price * obj.quantity
        return format(value.quantize(Decimal("0.01")), ".2f")


class InvoiceSerializer(serpy.Serializer):
    invoice_id = serpy.StrField()
    issued_at = ISODateField()
    customer = InvoiceCustomerSerializer()
    status = EnumValueField()
    items = InvoiceItemSerializer(many=True)
    subtotal = serpy.MethodField()
    tax = serpy.MethodField()
    total = serpy.MethodField()

    @staticmethod
    def _subtotal_value(obj: Invoice) -> Decimal:
        return sum(
            (item.unit_price * item.quantity for item in obj.items),
            start=Decimal("0"),
        )

    def get_subtotal(self, obj: Invoice) -> str:
        return format(
            self._subtotal_value(obj).quantize(Decimal("0.01")),
            ".2f",
        )

    def get_tax(self, obj: Invoice) -> str:
        tax = self._subtotal_value(obj) * obj.tax_rate
        return format(tax.quantize(Decimal("0.01")), ".2f")

    def get_total(self, obj: Invoice) -> str:
        subtotal = self._subtotal_value(obj)
        tax = subtotal * obj.tax_rate
        total = subtotal + tax
        return format(total.quantize(Decimal("0.01")), ".2f")


In [ ]:
invoice = Invoice(
    invoice_id="INV-2026-0001",
    issued_at=datetime(2026, 8, 7, 18, 0, tzinfo=timezone.utc),
    customer=InvoiceCustomer(
        id=300,
        name="Acme Example Ltd.",
        billing_email="billing@example.com",
    ),
    items=[
        InvoiceItem(
            sku="CONSULT-01",
            description="Architecture consulting",
            quantity=3,
            unit_price=Decimal("125.00"),
        ),
        InvoiceItem(
            sku="REVIEW-01",
            description="Code review",
            quantity=2,
            unit_price=Decimal("80.00"),
        ),
    ],
    tax_rate=Decimal("0.20"),
    status=PaymentStatus.UNPAID,
    internal_notes="Never expose this field through the public API.",
)

invoice_payload = InvoiceSerializer(invoice).data

print(json.dumps(invoice_payload, indent=2))

assert invoice_payload == {
    "invoice_id": "INV-2026-0001",
    "issued_at": "2026-08-07T18:00:00+00:00",
    "customer": {
        "id": 300,
        "name": "Acme Example Ltd.",
    },
    "status": "unpaid",
    "items": [
        {
            "sku": "CONSULT-01",
            "description": "Architecture consulting",
            "quantity": 3,
            "unit_price": "125.00",
            "subtotal": "375.00",
        },
        {
            "sku": "REVIEW-01",
            "description": "Code review",
            "quantity": 2,
            "unit_price": "80.00",
            "subtotal": "160.00",
        },
    ],
    "subtotal": "535.00",
    "tax": "107.00",
    "total": "642.00",
}

assert "internal_notes" not in invoice_payload
assert "billing_email" not in invoice_payload["customer"]
assert json.loads(json.dumps(invoice_payload)) == invoice_payload


## Problem 29 — Capstone extension: an API envelope

Serpy serializes the resource body. Add pagination or metadata **outside** the resource serializer so the serializer stays focused.

Build this envelope:

```json
{
  "data": [...],
  "meta": {
    "count": 2,
    "api_version": "v1"
  }
}
```


In [ ]:
invoice_2 = Invoice(
    invoice_id="INV-2026-0002",
    issued_at=datetime(2026, 8, 8, 9, 0, tzinfo=timezone.utc),
    customer=invoice.customer,
    items=[
        InvoiceItem(
            sku="SUPPORT-01",
            description="Support package",
            quantity=1,
            unit_price=Decimal("200.00"),
        )
    ],
    tax_rate=Decimal("0.20"),
    status=PaymentStatus.PAID,
    internal_notes="Internal only.",
)

serialized_invoices = InvoiceSerializer(
    [invoice, invoice_2],
    many=True,
).data

response_envelope = {
    "data": serialized_invoices,
    "meta": {
        "count": len(serialized_invoices),
        "api_version": "v1",
    },
}

print(json.dumps(response_envelope, indent=2))

assert response_envelope["meta"]["count"] == 2
assert response_envelope["data"][1]["status"] == "paid"


# Part 10 — Extra challenge problems


## Problem 30 — Challenge: flatten a nested value

Given:

```python
employee.department.name
```

emit:

```json
{"employee": "...", "department": "..."}
```

Use `attr=` rather than a `MethodField` because no computation is necessary.


In [ ]:
@dataclass
class Department:
    name: str


@dataclass
class Employee:
    name: str
    department: Department


class EmployeeSerializer(serpy.Serializer):
    employee = serpy.StrField(attr="name")
    department = serpy.StrField(attr="department.name")


employee = Employee("Sam", Department("Platform"))
employee_data = EmployeeSerializer(employee).data

print(employee_data)

assert employee_data == {
    "employee": "Sam",
    "department": "Platform",
}


## Problem 31 — Challenge: optional nested object

A user may or may not have a profile. Because a nested serializer is itself a field, mark it `required=False`.

Test both a real profile and `None`.


In [ ]:
@dataclass
class PublicProfile:
    bio: str


@dataclass
class MaybeProfileUser:
    id: int
    profile: PublicProfile | None


class PublicProfileSerializer(serpy.Serializer):
    bio = serpy.StrField()


class MaybeProfileUserSerializer(serpy.Serializer):
    id = serpy.IntField()
    profile = PublicProfileSerializer(required=False)


profile_users = [
    MaybeProfileUser(1, PublicProfile("Hello")),
    MaybeProfileUser(2, None),
]

profile_user_data = MaybeProfileUserSerializer(
    profile_users,
    many=True,
).data

print(profile_user_data)

assert profile_user_data == [
    {"id": 1, "profile": {"bio": "Hello"}},
    {"id": 2, "profile": None},
]


## Problem 32 — Challenge: prove serializers are output schemas, not object dumps

Add a new private attribute to `Person` dynamically, then show that the serializer output does not change.


In [ ]:
person_with_private_data = Person("Dana", 40)
person_with_private_data.ssn = "000-00-0000"
person_with_private_data.debug_token = "secret"

safe_person = PersonSerializer(person_with_private_data).data
print(safe_person)

assert safe_person == {
    "name": "Dana",
    "age": 40,
}
assert "ssn" not in safe_person
assert "debug_token" not in safe_person


# Final checklist — Serpy best practices

1. **Remember the direction:** Serpy serializes; it does not deserialize/validate inbound payloads.
2. **Keep serializers explicit:** declare public fields rather than dumping object internals.
3. **Use `many=True` deliberately** for collections.
4. **Use nested serializers** for structured child objects.
5. **Use `attr=`** for source mapping and dotted attribute access.
6. **Use `label=`** for output-key renaming.
7. **Use `call=True`** only for zero-argument model methods.
8. **Use `MethodField`** for calculated or presentation-oriented values.
9. **Use custom fields** for reusable scalar conversions such as dates, decimals, enums, and masking.
10. **Understand `required=False`:** missing values may be omitted; `None` is generally preserved without conversion.
11. **Avoid circular graphs:** choose finite API boundaries and serialize identifiers/summaries where appropriate.
12. **Keep format encoding separate:** Serpy -> native data -> JSON/YAML/web response.
13. **Handle money carefully:** prefer `Decimal` in the domain and deliberate string/decimal encoding at the boundary.
14. **Do not reuse a serializer instance after source mutation** if `.data` has already been accessed, because `.data` is cached.
15. **Benchmark your real workload:** object shape and JSON encoding may matter more than serializer micro-benchmarks.
16. **Pin old dependencies** when reproducibility matters, and reassess whether a maintained alternative better fits new projects.


## Further experiments

Try these without looking at the earlier solutions:

- Add a `URLField` that normalizes a hostname to lowercase.
- Add a `DurationSecondsField` for `datetime.timedelta`.
- Serialize a tree only to a maximum depth of one level.
- Build separate `PublicUserSerializer` and `AdminUserSerializer` schemas.
- Compare Serpy against a maintained serializer on your actual domain objects.
- Add `pytest` tests around every public serializer contract.
- Generate a JSON fixture and verify it with snapshot testing.
